In [2]:
!pip install rank_bm25


In [ ]:
import pandas as pd
from rank_bm25 import *
import warnings
warnings.filterwarnings('ignore')
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
sentence = "Jack is a sharp minded fellow"
words = word_tokenize(sentence)
print(words)

In [3]:
# read the contents of input.txt into the variable `string`
with open('input.txt', 'r', encoding='utf-8') as f:
    string = f.read()

In [5]:
len(string)

92455

In [6]:
# split the `string` into chunks of 200 words
words = string.split()
chunk_size = 200    
chunks_200 = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
print(f"Created {len(chunks_200)} chunks; first chunk words = {len(chunks_200[0].split()) if chunks_200 else 0}")

Created 72 chunks; first chunk words = 200


In [7]:
chunks_200[0]

'Apollo.io Email address Password Remember me EX-10.4 4 ex10-4.htm SUREN AJJARAPU EMPLOYMENT AGREEMENT Exhibit 10.4 TRXADE GROUP, INC. EXECUTIVE EMPLOYMENT AGREEMENT THIS EXECUTIVE EMPLOYMENT AGREEMENT (this “Agreement”) is entered into this 14th day of April 2020, to be effective as of the Effective Date as defined below between Trxade Group, Inc., a Delaware corporation (the “Company”), and Suren Ajjarapu, an individual (the “Executive”) (each of the Company and Executive are referred to herein as a “Party”, and collectively referred to herein as the “Parties”). WITNESSETH: WHEREAS, the Executive currently serves as the Chief Executive Officer of the Company; WHEREAS, the Executive is currently party to an Executive Employment Agreement dated on or around May 15, 2013 with Trxade, Inc., a Florida corporation, the wholly-owned subsidiary of the Company (the “Prior Agreement”)1; and WHEREAS, the Company desires to replace and supersede the Prior Agreement with this Agreement and to con

In [9]:
import re
from rank_bm25 import BM25Okapi, BM25L, BM25Plus   # whichever variant you need
# Preprocessing function
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Preprocess all chunks
processed_chunks = [preprocess_text(chunk) for chunk in chunks_200]

# Tokenize chunks into words
tokenized_chunks = [chunk.split() for chunk in processed_chunks]

# Initialize BM25
bm25 = BM25Okapi(tokenized_chunks)

# Define your query
query = "executive employment agreement compensation"

# Preprocess and tokenize the query
processed_query = preprocess_text(query)
tokenized_query = processed_query.split()

# Get BM25 scores
scores = bm25.get_scores(tokenized_query)

# Get top results
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

# Display results
print("Top 5 most relevant chunks:")
for rank, idx in enumerate(top_indices, 1):
    print(f"\nRank {rank} (Score: {scores[idx]:.4f}):")
    print(f"Chunk {idx}: {chunks_200[idx][:200]}...")

Top 5 most relevant chunks:

Rank 1 (Score: 6.6809):
Chunk 9: 2020. 1.8. At Will Employment, Confidential Information, Invention Assignment and Arbitration Agreement. A required condition to the Company’s acceptance of this Agreement is the entry by the Executiv...

Rank 2 (Score: 5.7777):
Chunk 49: through the use of Electronic Delivery as a defense to the formation of a contract, and each such Party forever waives any such defense, except to the extent such defense relates to lack of authentici...

Rank 3 (Score: 5.6315):
Chunk 27: Page 8 of 15 Initials SA/____ 3.4.6 Executive also agrees to keep the Company advised of his home and business address for a period of two (2) years after termination of Executive’s employment hereund...

Rank 4 (Score: 5.5851):
Chunk 14: the Executive, but to not offer such benefits to other executives of the Company, in the event such benefits are not customarily made available to substantially all of is senior executives. For exampl...

Rank 5 (Score: 5

In [10]:
top_30=sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:30]
print("\nTop 30 most relevant chunks:")


Top 30 most relevant chunks:


In [12]:
top_chunks=[]
for rank, idx in enumerate(top_30, 1):
    top_chunks.append(chunks_200[idx])
top_chunks

['2020. 1.8. At Will Employment, Confidential Information, Invention Assignment and Arbitration Agreement. A required condition to the Company’s acceptance of this Agreement is the entry by the Executive into the At Will Employment, Confidential Information, Invention Assignment and Arbitration Agreement in the form of Exhibit A attached hereto. April 14, 2020 Executive Employment Agreement Suren Ajjarapu Page 3 of 15 Initials SA/____ ARTICLE II. COMPENSATION AND OTHER BENEFITS 2.1. Base Salary. So long as this Agreement remains in effect, for all services rendered by Executive hereunder and all covenants and conditions undertaken by the Parties pursuant to this Agreement, the Company shall pay, and Executive shall accept, as compensation, an annual base salary (“Base Salary”) of $300,000. The Base Salary shall be payable in regular installments in accordance with the normal payroll practices of the Company, in effect from time to time, but in any event no less frequently than on a mon

In [13]:
# Now we can use for futher more refinement via dense retrieval using sentence transformers or any other embedding model.

In [14]:
from sentence_transformers import SentenceTransformer, util
# load a sentence‑transformer model
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")
# embed the top chunks (these were pulled from the BM25 step)
chunk_embeddings = model.encode(top_chunks, convert_to_tensor=True)
print("Top chunks embedded.")
# if you also want to compare them to the original query
query_text = " ".join(tokenized_query)        # ['executive',...]
print("converting query to embedding...")
query_embedding = model.encode(query_text, convert_to_tensor=True)

# cosine‑similarity scores between query and each chunk
cos_scores = util.cos_sim(query_embedding, chunk_embeddings)[0]

# inspect a few highest‑scoring chunks
ranked = sorted(enumerate(cos_scores), key=lambda x: x[1], reverse=True)
print("top 5 by dense similarity:")
for i, score in ranked[:5]:
    print(f"chunk index {i}, score {score:.4f}")
    print(top_chunks[i][:200], "...\n")

# the `chunk_embeddings` tensor can be kept for any downstream task

c:\Users\tarun\anaconda3\envs\test_env_gpu\lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:560.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


top 5 by dense similarity:
chunk index 4, score 0.6851
and conditions set forth in the applicable Stock Incentive Plan or Equity Compensation Plan, or award agreement, as such may describe the rights and obligations upon termination of employment of Execu ...

chunk index 0, score 0.6848
2020. 1.8. At Will Employment, Confidential Information, Invention Assignment and Arbitration Agreement. A required condition to the Company’s acceptance of this Agreement is the entry by the Executiv ...

chunk index 19, score 0.6441
for benefits payable under applicable benefit plans in which Executive is entitled to participate pursuant to Section 2.6 hereof through the Termination Date, subject to and in accordance with the ter ...

chunk index 28, score 0.6115
Executive or not previously paid by the Company). 7.13. Section 409A and 457A Compliance. To the extent applicable, this Agreement is intended to meet the requirements of Section 409A and 457A of the  ...

chunk index 21, score 0.6082
entry 

In [16]:
ranked = sorted(enumerate(cos_scores), key=lambda x: x[1], reverse=True)
print("\nTop 20 most relevant chunks:")
top_chunks_dense=[]
for idx,score in ranked[:20]:
    top_chunks_dense.append(top_chunks[idx])
top_chunks_dense


Top 20 most relevant chunks:


['and conditions set forth in the applicable Stock Incentive Plan or Equity Compensation Plan, or award agreement, as such may describe the rights and obligations upon termination of employment of Executive. April 14, 2020 Executive Employment Agreement Suren Ajjarapu Page 7 of 15 Initials SA/____ 3.4.2 If Executive’s employment is terminated by Executive pursuant to Section 3.1.6 (Good Reason), or pursuant to Section 3.1.7 (without Cause by the Company), (a) Executive shall be entitled to continue to receive the salary at the rate in effect upon the Termination Date of employment for eighteen (18) months following the Termination Date, payable in accordance with the Company’s normal payroll practices and policies, as if Executive’s employment had not terminated; (b) Executive shall be entitled to the pro rata amount of any Discretionary Bonus and Performance Bonus which would be payable to Executive had he remained employed for an additional eighteen (18) months following the Terminat

In [19]:
from sentence_transformers import CrossEncoder

print("Loading cross-encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder model loaded.")

# Prepare pairs
query_chunk_pairs = [[query_text, chunk] for chunk in top_chunks_dense]

print("Scoring query-chunk pairs with cross-encoder...")
cross_encoder_scores = cross_encoder.predict(query_chunk_pairs)

# Rank results
ranked_cross = sorted(
    enumerate(cross_encoder_scores),
    key=lambda x: x[1],
    reverse=True
)

print("\nTop 10 chunks by cross-encoder score:")
for idx, score in ranked_cross[:10]:
    print(f"Chunk {idx}, Score: {score:.4f}")
    print(top_chunks_dense[idx][:200], "...\n")

Loading cross-encoder model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder model loaded.
Scoring query-chunk pairs with cross-encoder...

Top 10 chunks by cross-encoder score:
Chunk 1, Score: 7.0361
2020. 1.8. At Will Employment, Confidential Information, Invention Assignment and Arbitration Agreement. A required condition to the Company’s acceptance of this Agreement is the entry by the Executiv ...

Chunk 0, Score: 6.9826
and conditions set forth in the applicable Stock Incentive Plan or Equity Compensation Plan, or award agreement, as such may describe the rights and obligations upon termination of employment of Execu ...

Chunk 13, Score: 5.1243
through the use of Electronic Delivery as a defense to the formation of a contract, and each such Party forever waives any such defense, except to the extent such defense relates to lack of authentici ...

Chunk 18, Score: 4.7905
Apollo.io Email address Password Remember me EX-10.4 4 ex10-4.htm SUREN AJJARAPU EMPLOYMENT AGREEMENT Exhibit 10.4 TRXADE GROUP, INC. EXECUTIVE EMPLOYMENT AGREEMENT THIS EXE